# Lesson 08 — Yes or No? Classification
**NSW Software Engineering Stage 6 | Software Automation**

---
### Not every question has a number as the answer
So far, our AI has been predicting **numbers** (marks, wing sizes, prices).

But sometimes we need a **category** — "Is this email spam or not spam?" — "Will this student pass or fail?"

This is **Classification** — and today we'll learn how it works.

**Your Goals for Today:**
1. Understand the difference between regression and classification.
2. Learn how the **sigmoid function** converts any number into a probability.
3. Build a pass/fail classifier using **Logistic Regression**.

Take your time. Read the text, and click **Run** on each code box as you go down the page.

In [ ]:
# 👇 RUN THIS CELL FIRST 👇
!pip install numpy scikit-learn --prefer-binary --quiet

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

print("✅ Setup Complete! Move to the next section.")

---
## Concept 1: Regression vs Classification

| | Regression | Classification |
|---|---|---|
| **Output** | A number (56.3, 901.2...) | A category (0 or 1, pass or fail) |
| **Example question** | "What score will they get?" | "Will they pass or fail?" |
| **Algorithm** | Linear Regression | Logistic Regression |
| **Evaluation** | RMSE (error in units) | Accuracy (% correct) |

### The Problem with Linear Regression for Yes/No Questions
If we use linear regression to predict pass/fail, the AI might output 1.7 or -0.3 — which makes no sense!

We need outputs that are always **between 0 and 1** — a *probability*.

---
## Concept 2: The Sigmoid Function

The **sigmoid function** squashes any number into the range (0, 1):

> **σ(z) = 1 / (1 + e⁻ᶻ)**

| z value | σ(z) | Meaning |
|---|---|---|
| very negative | ≈ 0.0 | Almost certainly class 0 |
| 0 | 0.5 | 50/50 |
| very positive | ≈ 1.0 | Almost certainly class 1 |

**Decision rule:**
- If probability ≥ 0.5 → predict **1** (e.g., PASS)
- If probability < 0.5 → predict **0** (e.g., FAIL)

In [ ]:
# 👇 RUN THIS CELL 👇

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = [-6, -3, -1, 0, 1, 3, 6]
print("=== Sigmoid Function ===")
print(f"{'z':>6}  {'σ(z)':>8}  {'Prediction':>12}")
for z in z_values:
    s = sigmoid(z)
    pred = "Class 1 ✅" if s >= 0.5 else "Class 0 ❌"
    print(f"{z:>6}  {s:>8.4f}  {pred}")

---
## Concept 3: Logistic Regression — Step by Step

In [ ]:
# 👇 RUN THIS CELL 👇

# Will a student pass based on hours studied and assignment average?
np.random.seed(42)
n = 80
hours_c  = np.random.uniform(1, 10, n)
assign_c = np.random.uniform(50, 100, n)
# Pass if score combo > threshold
passes = ((5*hours_c + 0.3*assign_c + np.random.normal(0,5,n)) > 42).astype(int)

X_cls = np.column_stack([hours_c, assign_c])
y_cls = passes

# Split and scale
X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y_cls, test_size=0.2, random_state=7)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# Train — LogisticRegression, not LinearRegression!
clf = LogisticRegression()
clf.fit(X_tr_s, y_tr)

# Evaluate
y_pred = clf.predict(X_te_s)
acc    = accuracy_score(y_te, y_pred)
print(f"✅ Pass/Fail Classifier trained!")
print(f"   Accuracy: {acc:.0%}  ({int(acc*len(y_te))}/{len(y_te)} correct)")

# Predict with probabilities
print("\n=== Predictions for 3 new students ===")
new_students = [[3, 60], [7, 85], [5, 72]]
new_scaled   = scaler.transform(new_students)
probs        = clf.predict_proba(new_scaled)

print(f"{'Hours':>6} {'Assign':>8} {'P(Pass)':>10} {'Verdict':>10}")
for (h,a), p in zip(new_students, probs):
    verdict = "PASS ✅" if p[1] >= 0.5 else "FAIL ❌"
    print(f"{h:>6} {a:>8} {p[1]:>10.2%} {verdict:>10}")

---
## ✏️ Exercise 1: Build a Simple Classifier

Fill in the `____` blanks to build a classifier that predicts whether a bug is **Species 0 or 1** based on wing size.

In [ ]:
# 👇 FILL IN THE BLANKS AND RUN THIS CELL 👇

bug_wings   = np.array([[850],[870],[880],[900],[910],[920],[935],[950]])
bug_species = np.array([0,    0,    0,    1,    1,    1,    1,    1   ])

# STEP 1: Create a LogisticRegression (not LinearRegression!)
# TODO: Replace ____ with LogisticRegression()
bug_clf = ____

# STEP 2: Train the classifier
# TODO: Fill in the inputs and labels
bug_clf.fit(____, ____)

# STEP 3: Predict the species for an 895mm wing
# TODO: Replace ____ with [[895]]
prediction = bug_clf.predict(____)
probability = bug_clf.predict_proba([[895]])[0]

# ⬇️ DON'T EDIT BELOW THIS LINE — THIS IS THE AUTOGRADER ⬇️
print("=== Exercise 1 Results ===")
correct_pred = LogisticRegression().fit(bug_wings, bug_species).predict([[895]])[0]
try:
    assert prediction[0] == correct_pred
    print(f"✅ Prediction for 895mm: Species {prediction[0]}")
    print(f"   P(Species 0) = {probability[0]:.2%}")
    print(f"   P(Species 1) = {probability[1]:.2%}")
except: print(f"❌ Prediction: got {prediction}, expected {correct_pred}")

---
## ✏️ Exercise 2: Confusion Matrix Vocabulary

A **Confusion Matrix** tells us exactly how the model made mistakes.

```
                     Predicted
                  FAIL(0)  PASS(1)
Actual  FAIL(0)  [  TN       FP  ]
Actual  PASS(1)  [  FN       TP  ]
```

Replace `None` with `"TP"`, `"TN"`, `"FP"`, or `"FN"`.

In [ ]:
# 👇 FILL IN THE BLANKS AND RUN THIS CELL 👇

cm_quiz = {
    # Example:
    "Student actually passed AND the model said PASS": "TP",

    # TODO: Fill in the 3 blanks:
    "Student actually failed AND the model said FAIL": None,
    "Student actually passed BUT the model said FAIL (missed them!)": None,
    "Student actually failed BUT the model said PASS (false alarm!)": None,
}

# ⬇️ DON'T EDIT BELOW THIS LINE — THIS IS THE AUTOGRADER ⬇️
correct = {
    "Student actually passed AND the model said PASS": "TP",
    "Student actually failed AND the model said FAIL": "TN",
    "Student actually passed BUT the model said FAIL (missed them!)": "FN",
    "Student actually failed BUT the model said PASS (false alarm!)": "FP",
}
score = 0
print("=== Exercise 2 Results ===")
for q, your_ans in cm_quiz.items():
    if q == "Student actually passed AND the model said PASS": continue
    ans = correct[q]
    if your_ans == ans:
        print(f"  ✅ {ans} — {q[:60]}")
        score += 1
    elif your_ans is None:
        print(f"  ⬜ Not answered")
    else:
        print(f"  ❌ You said '{your_ans}', correct is '{ans}'")
print(f"\nFinal Score: {score}/3")

---
## 🚀 EXTENSION CHALLENGES 🚀

### Exercise 3: Classify Iris Flowers

The famous Iris dataset has 3 species of flower. Let's simplify it to binary:
**Is it Setosa (1) or not (0)?**

In [ ]:
# 👇 FILL IN THE BLANKS AND RUN THIS CELL 👇
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data         # 4 features: sepal/petal length and width
y_iris = (iris.target == 0).astype(int)   # 1 = setosa, 0 = not setosa

# STEP 1: Split 80/20
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)

# STEP 2: Scale
sc_i = StandardScaler()
X_tr_i_s = sc_i.fit_transform(____)     # TODO: fill in X_tr_i
X_te_i_s = sc_i.transform(____)         # TODO: fill in X_te_i

# STEP 3: Train LogisticRegression
clf_iris = LogisticRegression()
clf_iris.fit(____, y_tr_i)              # TODO: fill in scaled train X

# STEP 4: Accuracy and confusion matrix
y_pred_i = clf_iris.predict(X_te_i_s)
acc_i = accuracy_score(____, y_pred_i)  # TODO: fill in y_te_i

print(f"✅ Iris Setosa Classifier Accuracy: {acc_i:.0%}")
print(f"   (Expected: ~100% — Setosa is very easy to separate!)")
print(f"\nConfusion matrix:\n{confusion_matrix(y_te_i, y_pred_i)}")

### Exercise 4: The Debugging Challenge

The code below tries to build a classifier, but has **TWO bugs** — one in training, one in prediction.
Fix them.

In [ ]:
# 👇 FIX THE TWO BUGS IN THIS CODE 👇

X_bug = np.array([[850],[870],[900],[920],[940]])
y_bug = np.array([0, 0, 1, 1, 1])

scaler_bug = StandardScaler()
X_bug_s    = scaler_bug.fit_transform(X_bug)

# BUG 1: Using LinearRegression instead of LogisticRegression for classification
from sklearn.linear_model import LinearRegression
clf_bug = LinearRegression()
clf_bug.fit(X_bug_s, y_bug)

# BUG 2: Predicting without scaling the new input
new_wing   = [[905]]
prediction = clf_bug.predict(new_wing)   # ← missing scaler step!

print(f"Species prediction: {prediction[0]:.2f}")

### Exercise 5: Evaluate Your Own Classifier 🏆

Build a **complete classifier pipeline** to predict whether an insect is **male (1) or female (0)** based on `latitude` and `wingsize`.

Your code must:
1. Load `insects.csv`, use `latitude` and `wingsize` as features, `sex` as label
2. Split 80/20, scale features
3. Train `LogisticRegression`
4. Print accuracy AND the full confusion matrix
5. Predict the sex for latitude=42, wingsize=905

In [ ]:
# 👇 WRITE ALL THE CODE YOURSELF IN THIS CELL 👇
import pandas as pd
df = pd.read_csv('insects.csv', sep='\t')

# Step 1: Prepare X and y


# Step 2: Split and scale


# Step 3: Train LogisticRegression


# Step 4: Print accuracy and confusion matrix


# Step 5: Predict for latitude=42, wingsize=905




---
## ✅ Lesson 8 Complete!
You can now build classifiers — AI that predicts categories. This is how spam filters, medical AI, and fraud detection systems work.

**Next up:** Lesson 09 — Is My AI Too Smart? We'll learn about overfitting and how to spot when an AI is memorising instead of learning.

---









































## 🔑 Answer Key

### Exercise 1
```python
bug_clf = LogisticRegression()
bug_clf.fit(bug_wings, bug_species)
prediction = bug_clf.predict([[895]])
```

### Exercise 2
```
"actually failed, said FAIL" → "TN"
"actually passed, said FAIL" → "FN"
"actually failed, said PASS" → "FP"
```

### Exercise 3
```python
X_tr_i_s = sc_i.fit_transform(X_tr_i)
X_te_i_s = sc_i.transform(X_te_i)
clf_iris.fit(X_tr_i_s, y_tr_i)
acc_i = accuracy_score(y_te_i, y_pred_i)
```

### Exercise 4 — Two bugs fixed:
```python
# BUG 1: Use LogisticRegression for classification
clf_bug = LogisticRegression()
# BUG 2: Scale the new input before predicting
new_wing_s = scaler_bug.transform([[905]])
prediction = clf_bug.predict(new_wing_s)
```

### Exercise 5 — Sample answer:
```python
X=df[['latitude','wingsize']].values; y=df['sex'].values
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)
sc=StandardScaler(); Xtr_s=sc.fit_transform(Xtr); Xte_s=sc.transform(Xte)
clf=LogisticRegression().fit(Xtr_s,ytr)
print(f"Accuracy: {accuracy_score(yte,clf.predict(Xte_s)):.0%}")
print(confusion_matrix(yte,clf.predict(Xte_s)))
new=sc.transform([[42,905]]); print(f"Sex: {clf.predict(new)[0]}")
```
